# Codebase Agent

Week 2: turning the week 1 RAG pipeline (`02-rag.ipynb`) into an agent that calls tools, following the course's `documentation-agent` (`tools.py` + `doc_agent.py`).

The agent investigates the target repo ([serverless-eda-insurance-claims-processing](https://github.com/aws-samples/serverless-eda-insurance-claims-processing)) and answers how part of the system works, citing `file:line`. Run `make targets` first.

Tools in this version:

1. `list_files` - see what's in the repo
2. `search_code` - exact / regex search over the code
3. `read_file` - read numbered lines from a file
4. `search_docs` - the week 1 `search` over READMEs and EventCatalog

`describe_table`, `get_event_flow` and `list_endpoints` come next.

In [1]:
import fnmatch
import json
import re
from pathlib import Path

from dotenv import load_dotenv
load_dotenv()

import frontmatter
from gitsource import chunk_documents
from minsearch import Index
from pydantic_ai import Agent
from pydantic_ai.messages import FunctionToolCallEvent

from safe_change_agent.config import get_model

## 1. Data

Same loading as week 1, but the search index now holds **only the docs**. The code is reached through `search_code` and `read_file`, which are exact and give real line numbers.

In [2]:
REPO_DIR = Path("../targets/insurance-claims").resolve()

SKIP_DIRS = {"node_modules", ".git", "cdk.out", ".github"}
DOC_EXTENSIONS = {".md", ".mdx"}
CODE_EXTENSIONS = {".ts", ".py", ".js", ".json", ".yaml", ".yml"}


def is_skipped(rel_path):
    return any(part in SKIP_DIRS for part in rel_path.parts)


def as_text(value):
    if isinstance(value, list):
        return ", ".join(str(v) for v in value)
    return str(value) if value else ""


def load_docs(repo_dir):
    documents = []
    for path in sorted(repo_dir.rglob("*")):
        rel = path.relative_to(repo_dir)
        if not path.is_file() or is_skipped(rel) or path.suffix not in DOC_EXTENSIONS:
            continue
        post = frontmatter.load(path)
        documents.append({
            "filename": str(rel),
            "title": as_text(post.metadata.get("name") or post.metadata.get("title") or path.parent.name),
            "summary": as_text(post.metadata.get("summary")),
            "producers": as_text(post.metadata.get("producers")),
            "consumers": as_text(post.metadata.get("consumers")),
            "content": post.content,
        })
    return documents


def build_docs_index(documents):
    contents = {d["filename"]: d["content"] for d in documents}
    chunks = chunk_documents(documents, size=2000, step=1000)
    for chunk in chunks:
        chunk["start_line"] = contents[chunk["filename"]][:chunk["start"]].count("\n") + 1

    index = Index(
        text_fields=["title", "summary", "producers", "consumers", "content", "filename"],
        keyword_fields=["filename"],
    )
    index.fit(chunks)
    return index


docs = load_docs(REPO_DIR)
docs_index = build_docs_index(docs)
len(docs)

43

## 2. Tools

Like `SearchTools` in the course: one class, each tool a method with type hints and a docstring. PydanticAI turns the docstring into the tool description the model sees, so the docstrings say *when* to use each tool.

Every path is resolved and checked to stay inside the target repo.

In [3]:
class CodebaseTools:
    """Read-only tools for investigating a codebase."""

    def __init__(self, repo_dir: Path, docs_index: Index, max_results: int = 50, max_lines: int = 200):
        self.repo_dir = repo_dir.resolve()
        self.docs_index = docs_index
        self.max_results = max_results
        self.max_lines = max_lines

    def _resolve(self, path: str) -> Path:
        full = (self.repo_dir / path).resolve()
        if not full.is_relative_to(self.repo_dir):
            raise ValueError(f"path outside the repo: {path}")
        return full

    def _code_files(self, root: Path):
        for path in sorted(root.rglob("*")):
            rel = path.relative_to(self.repo_dir)
            if path.is_file() and not is_skipped(rel) and path.suffix in CODE_EXTENSIONS:
                yield path, rel

    def list_files(self, directory: str = ".", pattern: str | None = None) -> list[str]:
        """
        List files in the repo under a directory. Call this first to see which
        services exist and where their infrastructure (infra/) and handler (app/) code lives.

        Args:
            directory: Directory relative to the repo root, e.g. "lib/services/claims".
            pattern: Optional glob on the file name, e.g. "*.ts".

        Returns:
            Relative file paths (max 200).
        """
        try:
            root = self._resolve(directory)
        except ValueError as e:
            return [f"Error: {e}"]

        files = []
        for path in sorted(root.rglob("*")):
            rel = path.relative_to(self.repo_dir)
            if not path.is_file() or is_skipped(rel):
                continue
            if pattern and not fnmatch.fnmatch(path.name, pattern):
                continue
            files.append(str(rel))
        return files[:200]

    def search_code(self, pattern: str, path_glob: str | None = None) -> list[str]:
        """
        Search the code with a case-insensitive regex. Use it to find where an
        event, field, table, type or function is defined or used. Event names
        appear in different forms in code, so search variations, e.g.
        "Claim.?Accepted|CLAIM_ACCEPTED".

        Args:
            pattern: Python regular expression.
            path_glob: Optional glob on the relative path, e.g. "lib/services/claims/*".

        Returns:
            Matches as "file:line: text" (max 50).
        """
        try:
            regex = re.compile(pattern, re.IGNORECASE)
        except re.error as e:
            return [f"Error: invalid regex: {e}"]

        matches = []
        for path, rel in self._code_files(self.repo_dir):
            if path_glob and not fnmatch.fnmatch(str(rel), path_glob):
                continue
            try:
                lines = path.read_text(encoding="utf-8").splitlines()
            except UnicodeDecodeError:
                continue
            for i, line in enumerate(lines, start=1):
                if regex.search(line):
                    matches.append(f"{rel}:{i}: {line.strip()[:200]}")
                    if len(matches) >= self.max_results:
                        matches.append("... more matches, narrow the pattern or path_glob")
                        return matches
        return matches

    def read_file(self, path: str, start_line: int = 1, end_line: int | None = None) -> str:
        """
        Read lines from a file, prefixed with line numbers. Use it after
        search_code or list_files to read the actual code before making claims,
        and cite what you read as file:line.

        Args:
            path: File path relative to the repo root.
            start_line: First line to read (1-based).
            end_line: Last line to read. At most 200 lines are returned per call.

        Returns:
            Numbered lines, or an error message.
        """
        try:
            full = self._resolve(path)
        except ValueError as e:
            return f"Error: {e}"
        if not full.is_file():
            return f"Error: file {path} does not exist"

        lines = full.read_text(encoding="utf-8", errors="replace").splitlines()
        start = max(start_line, 1)
        end = min(end_line or len(lines), start + self.max_lines - 1, len(lines))

        numbered = [f"{i}: {lines[i - 1]}" for i in range(start, end + 1)]
        if end < len(lines):
            numbered.append(f"... file has {len(lines)} lines, read more with start_line={end + 1}")
        return "\n".join(numbered)

    def search_docs(self, query: str) -> list[dict]:
        """
        Search the repo's documentation (READMEs and EventCatalog pages for events
        and services, with their producers and consumers). Use it to see what the
        docs claim, then verify those claims in the code - docs can be outdated.

        Args:
            query: Natural language search query.

        Returns:
            Matching doc chunks with filename, start_line, producers, consumers and content.
        """
        results = self.docs_index.search(
            query,
            boost_dict={"title": 2.0, "producers": 1.5, "consumers": 1.5},
            num_results=5,
        )
        return [
            {k: r[k] for k in ["filename", "start_line", "title", "producers", "consumers", "content"]}
            for r in results
        ]


tools = CodebaseTools(REPO_DIR, docs_index)

Quick check of each tool before handing them to the agent:

In [4]:
print(tools.list_files("lib/services", "*-service.ts"))
print(tools.search_code("Claim.?Accepted|CLAIM_ACCEPTED")[:5])
print(tools.read_file("lib/services/claims/infra/claims-events.ts"))
print([r["filename"] for r in tools.search_docs("claim accepted")])
print(tools.read_file("../../etc/passwd"))

['lib/services/claims/infra/claims-service.ts', 'lib/services/customer/infra/customer-service.ts', 'lib/services/documents/infra/documents-service.ts', 'lib/services/fraud/infra/fraud-service.ts', 'lib/services/frontend/infra/frontend-hosting-service.ts', 'lib/services/notifications/infra/notifications-service.ts', 'lib/services/settlement/infra/settlement-service.ts', 'lib/services/vendor/infra/vendor-service.ts', 'lib/services/voice-fnol-agent/infra/voice-fnol-service.ts']
['event-catalog/events/ClaimAccepted/schema.json:5: "title": "Claim.Accepted"', 'event-catalog/events/ClaimAccepted/schema.json:13: "x-amazon-events-detail-type": "Claim.Accepted",', 'event-catalog/events/ClaimAccepted/schema.json:17: "$ref": "#/components/schemas/Claim.Accepted"', 'event-catalog/events/ClaimAccepted/schema.json:49: "Claim.Accepted": {', 'lib/claims-processing-stack.ts:118: ClaimsEvents.CLAIM_ACCEPTED,']
1: export enum ClaimsEvents {
2:   FNOL_SOURCE = "fnol.service",
3:   CLAIMS_SOURCE = "claims.s

## 3. Agent

Instructions describe the job and when to use each tool. This replaces `build_prompt` + `instructions` from week 1: the agent decides what context to fetch.

In [5]:
instructions = """
You help engineers understand an unfamiliar AWS serverless codebase before they
change it. The system is an event-driven insurance claims app built with CDK:
Lambda, EventBridge, SQS, Step Functions, DynamoDB, API Gateway, Cognito.

How to investigate:
1. Orient: use list_files to find the relevant service(s). Each service lives in
   lib/services/<name>/ with infra/ (CDK) and app/ (handlers).
2. Check the docs: use search_docs to see what the docs say about the topic.
3. Find it in the code: use search_code with several spellings of a name, because
   code uses constants like CLAIM_ACCEPTED = "Claim.Accepted" rather than "ClaimAccepted".
   Look for where things are defined AND where they are used (EventBridge rules,
   event patterns, putEvents calls, table writes).
4. Read before you claim: use read_file on the lines you found. Never describe
   code you haven't read.

How to answer:
- Explain the flow step by step: who produces what, who consumes it, what each step does.
- Cite every claim as `path:line` from what you read.
- The code is the source of truth. If the docs disagree with the code, say so
  explicitly and cite both.
- If you couldn't find something, say what you searched for and where you would look next.
- Be concise. Use short sections and bullet points.
""".strip()


agent = Agent(
    name="codebase_agent",
    model=get_model(),
    instructions=instructions,
    tools=[tools.list_files, tools.search_code, tools.read_file, tools.search_docs],
)

Print each tool call as it happens, like `NamedCallback` in `doc_agent.py`:

In [6]:
async def print_tool_calls(ctx, events):
    async for event in events:
        if isinstance(event, FunctionToolCallEvent):
            print(f"TOOL CALL: {event.part.tool_name}({event.part.args})")


async def ask(question):
    print(f"QUESTION: {question}\n")
    result = await agent.run(question, event_stream_handler=print_tool_calls)
    usage = result.usage
    print(f"\n(requests: {usage.requests}, input tokens: {usage.input_tokens}, output tokens: {usage.output_tokens})\n")
    print(result.output)
    return result

## 4. Try It Out

The week 1 question. RAG only found the docs; the agent should find the code too.

In [7]:
result = await ask("What happens after a claim is accepted? Which services consume the ClaimAccepted event and what do they do?")

QUESTION: What happens after a claim is accepted? Which services consume the ClaimAccepted event and what do they do?



TOOL CALL: search_code({"pattern":"Claim.Accepted"})


TOOL CALL: list_files({"directory":"lib/services/claims"})


TOOL CALL: read_file({"path":"lib/services/claims/app/handlers/claimsProcessing.js","start_line":20,"end_line":40})
TOOL CALL: read_file({"path":"lib/services/claims/app/handlers/claimsProcessing.js","start_line":150,"end_line":180})
TOOL CALL: read_file({"path":"lib/services/claims/infra/claims-events.ts","start_line":0,"end_line":20})
TOOL CALL: read_file({"path":"lib/services/claims/infra/claims-service.ts","start_line":240,"end_line":260})
TOOL CALL: read_file({"path":"lib/services/claims/README.md","start_line":0,"end_line":20})



(requests: 4, input tokens: 5961, output tokens: 759)

### Overview of Claim Acceptance Process

When a claim is accepted, a series of events trigger actions within the insurance claims application. 

#### Steps After Claim is Accepted

1. **Event Emission**:
   - The claims processing Lambda function emits a `Claim.Accepted` event after processing the claim successfully. This event is defined in the `ClaimsEvents` enum (see `lib/services/claims/infra/claims-events.ts:4`).

2. **Event Creation**:
   - In the `claimsProcessing.js` handler, after validating the claim details and processing any necessary uploads (like creating a pre-signed URL for uploading images), the function constructs the `eventPayload` with the following details:
     - `detailType`: "Claim.Accepted"
     - Additional information such as `claimId` and `uploadCarDamageUrl` to provide further context to any consumer of this event (see `lib/services/claims/app/handlers/claimsProcessing.js:158`).

3. **Publishing the E

A data question, one of the kinds I ask at work:

In [8]:
result = await ask("Which DynamoDB table stores customer policies, what are its keys, and which code writes to it?")

QUESTION: Which DynamoDB table stores customer policies, what are its keys, and which code writes to it?



TOOL CALL: list_files({"directory":"lib/services"})


TOOL CALL: search_docs({"query":"customer policies"})
TOOL CALL: search_code({"pattern":"policy","path_glob":"lib/services/customer/*"})
TOOL CALL: search_code({"pattern":"Policy","path_glob":"lib/services/customer/*"})


TOOL CALL: read_file({"path":"lib/services/customer/infra/customer-service.ts","start_line":99,"end_line":113})


TOOL CALL: read_file({"path":"lib/services/customer/app/handlers/putPolicyRequests.js","start_line":20,"end_line":30})


TOOL CALL: read_file({"path":"lib/services/customer/app/handlers/putPolicyRequests.js","start_line":30,"end_line":40})



(requests: 6, input tokens: 25031, output tokens: 685)

### DynamoDB Table for Customer Policies

- **Table Name**: The DynamoDB table that stores customer policies is called **PolicyTable**.

### Table Keys

- **Partition Key (PK)**: A unique identifier for the policy, generated using `uuidv4()` (e.g., `policyId`).
- **Sort Key (SK)**: A string formatted as `Customer|{uuid}`, where `{uuid}` is the unique identifier of the customer.

The table structure is defined as follows:
```typescript
this.policyTable = new Table(this, "PolicyTable", {
    billingMode: BillingMode.PAY_PER_REQUEST,
    partitionKey: { name: "PK", type: AttributeType.STRING },
    sortKey: { name: "SK", type: AttributeType.STRING },
    removalPolicy: RemovalPolicy.DESTROY,
});
```
*(See: `lib/services/customer/infra/customer-service.ts:99-104`)*

### Code Writing to the Table

The following code writes to the PolicyTable:

1. **Handler Function**: 
   - **File**: `putPolicyRequests.js`
   - **Key Method**:
     - 

A docs question, which should go mainly through `search_docs`:

In [9]:
result = await ask("According to the docs, what is the FraudDetected event for and which services produce and consume it? Does the code agree?")

QUESTION: According to the docs, what is the FraudDetected event for and which services produce and consume it? Does the code agree?



TOOL CALL: search_docs({"query":"FraudDetected event"})


TOOL CALL: search_code({"pattern":"FraudDetected"})



(requests: 3, input tokens: 6568, output tokens: 325)

### FraudDetected Event Documentation

- **Purpose**: The FraudDetected event is triggered by the **Fraud Service** when fraud is detected in a customer document that has been uploaded.
- **Producers**: 
  - **Fraud Service** (This service creates the event when it detects fraud.)
- **Consumers**: 
  - **Notifications Service** (This service uses the event to notify the customer about the status of their request.)

### Code Verification

I searched the code for "FraudDetected," but I did not find any references in the codebase. This could mean that:

- The event might be defined in a part of the repository that isn't accessible or hasn't been implemented yet.
- There might be a different naming convention used in the codebase (e.g., constants or prefixes).

### Next Steps

1. **Further Searches**: 
   - I would recommend searching for variations of the event name, such as:
     - `FRAUD_DETECTED`
     - `FraudDetectedEvent`
     -

An edge case: there is no payment or refund service in this repo. The agent should say it can't find it, not invent one.

In [10]:
result = await ask("How does the payment service refund the customer after a claim is rejected?")

QUESTION: How does the payment service refund the customer after a claim is rejected?



TOOL CALL: list_files({"directory":"lib/services/payment"})


TOOL CALL: list_files({"directory":"lib/services/claims"})


TOOL CALL: search_docs({"query":"refund after a claim is rejected"})
TOOL CALL: search_code({"pattern":"refund|Refund","path_glob":"lib/services/payment/*"})
TOOL CALL: search_code({"pattern":"ClaimRejected|CLAIM_REJECTED|claim_rejected","path_glob":"lib/services/claims/*"})


TOOL CALL: read_file({"path":"lib/services/claims/infra/claims-events.ts","start_line":1,"end_line":20})
TOOL CALL: read_file({"path":"lib/services/claims/infra/claims-service.ts","start_line":240,"end_line":260})


TOOL CALL: search_code({"pattern":"refund|Refund","path_glob":"lib/services/payment/*"})



(requests: 6, input tokens: 9526, output tokens: 697)

### Flow of Refund Process after Claim Rejection

1. **Claim Rejection Event**:
   - When a claim is rejected, the **Claims Service** emits a `ClaimRejected` event. This is defined in the event catalog and indicated in the code with:
     ```typescript
     CLAIM_REJECTED = "Claim.Rejected"
     ```
     (found in `lib/services/claims/infra/claims-events.ts:5`).

2. **Triggering the Refund**:
   - The refund process would typically be initiated by a listener or consumer of the `ClaimRejected` event, responsible for handling the payment logic. However, specifics for refund operations weren't found in the Payment Service's codebase or documentation.

3. **Notifications**:
   - The `ClaimRejected` event also triggers notifications to inform the customer about the rejection. The documentation indicates that the **Notifications Service** is a consumer of this event:
     - `"producers": "Claims Service", "consumers": "Notifications Ser